# Training DVAD (DeepLabV3) untuk Bird Sound Denoising

Notebook ini mengikuti paper *DVAD: Deep Visual Audio Denoising* di mana kita mentransformasikan masalah denoising menjadi **Image Segmentation** (2-Stage Pipeline).

**Langkah-langkah Pipeline DVAD:**
1. Konversi *audio waveform* menjadi gambar *spectrogram* (STFT) [Dataset sudah menyediakannya di Zenodo]
2. **Latih model Image Segmentation** (DeepLabV3) menggunakan gambar spectrogram dan label mask binary
3. **Inference/Denoising:** Prediksi mask dari gambar spectrogram, setel area noise di *complex STFT* ke 0, lalu gunakan Inverse STFT (ISTFT) untuk mengembalikan audio yang bersih.

Pastikan Runtime Google Colab diatur ke **GPU**.

## 1. Setup & Download Dataset
Dataset yang digunakan berasal dari dataset BirdSoundsDenoising.



In [ ]:
# Clone repositori Anda dan install dependency
!git clone https://github.com/user312982/bird-denoising-sound.git
%cd bird-denoising-sound
!pip install torchaudio torchvision matplotlib Pillow

import os
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

# Import custom modules
from models.dataset import BirdSoundsSegmentationDataset
from models.dvad import DVADSegmenter
from models.losses import CombinedLoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Menggunakan device: {device}")

## 2. Dataloader & Visualisasi Dataset

Dalam DVAD, dataset berupa gambar *Spectrogram* (Images) dan label *Binary Mask* (Masks).

In [ ]:
# Struktur Dataset Zenodo:
# dataset/Train/Images/
# dataset/Train/Masks/

train_img_dir = 'dataset/Train/Images'
train_mask_dir = 'dataset/Train/Masks'
val_img_dir = 'dataset/Valid/Images'
val_mask_dir = 'dataset/Valid/Masks'

# Resolusi DeepLabV3 sesuai paper
img_size = (512, 512)
batch_size = 8

train_dataset = BirdSoundsSegmentationDataset(
    images_dir=train_img_dir,
    masks_dir=train_mask_dir,
    img_size=img_size,
    augment=True
)

val_dataset = BirdSoundsSegmentationDataset(
    images_dir=val_img_dir,
    masks_dir=val_mask_dir,
    img_size=img_size,
    augment=False
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Total Sampel Train: {len(train_dataset)}")
print(f"Total Sampel Validasi: {len(val_dataset)}")

In [ ]:
# Visualisasikan satu sampel dataset
for images, masks in train_loader:
    plt.figure(figsize=(10, 5))
    
    # Denormalize gambar untuk visualisasi
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img_vis = images[0] * std + mean
    
    plt.subplot(1, 2, 1)
    plt.imshow(img_vis.permute(1, 2, 0).clip(0, 1))
    plt.title('Spectrogram Image')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(masks[0].numpy(), cmap='gray')
    plt.title('Ground Truth Mask\n(Putih = Clean, Hitam = Noise)')
    plt.axis('off')
    
    plt.show()
    break

## 3. Inisialisasi Model, Loss, dan Optimizer

Menggunakan model **DeepLabV3** dengan loss **Dice Loss + CrossEntropy** sesuai metodologi Paper.

In [ ]:
# 1. Model
model = DVADSegmenter(num_classes=2, pretrained_backbone=True).to(device)

# 2. Loss Function (Dice + CE)
criterion = CombinedLoss(dice_weight=0.5, ce_weight=0.5).to(device)

# 3. Optimizer (Adam seperti di paper)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# 4. Learning Rate Scheduler (opsional, untuk stabilisasi)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

epochs = 100  # Sesuai training iteration/epoch di paper

## 4. Training Loop Model Segmentasi

In [ ]:
best_val_loss = float('inf')
save_path = "best_dvad_segmenter.pth"

print("Mulai Training...")
for epoch in range(epochs):
    # --- TRAINING ---
    model.train()
    train_loss = 0.0
    
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass (Input image -> Logits)
        logits = model(images)
        
        # Calculate loss (Dice + CE)
        loss = criterion(logits, masks)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        
    train_loss = train_loss / len(train_dataset)
    
    # --- VALIDATION ---
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Valid]"):
            images = images.to(device)
            masks = masks.to(device)
            
            logits = model(images)
            loss = criterion(logits, masks)
            val_loss += loss.item() * images.size(0)
            
    val_loss = val_loss / len(val_dataset)
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), save_path)
        print(f"  [+] Model terbaik disimpan ke '{save_path}'")

print("Training Selesai!")

## 5. Denoising / Inference Process (Tahap ke-2)

Setelah model segmentasi dilatih, kita menggunakannya untuk men-denoise audio baru.
Sesuai paper, prosesnya adalah: `Wav -> STFT -> Prediksi Mask -> Apply Mask ke STFT -> ISTFT -> Clean Wav`

*(Kita belum memuat ulang fungsi ISTFT dari `models/audio_processing.py` secara lengkap di sini karena script testing/inference akan diurus terpisah, tapi kode di atas akan melatih model DeepLabV3 sesuai paper).*